In [14]:
from collections import Counter
from math import exp, log
from pathlib import Path

import pandas as pd

candidates = [
    Path.cwd() / "Lab 1" / "outputs" / "tokenized_sentences_with_source.txt",
    Path.cwd().parent / "Lab 1" / "outputs" / "tokenized_sentences_with_source.txt",
    Path(r"E:\STUDY\NLP\Lab 1\outputs\tokenized_sentences_with_source.txt"),
]
data_path = next((path for path in candidates if path.exists()), None)
if data_path is None:
    raise FileNotFoundError("Could not find tokenized_sentences_with_source.txt")

input_sentences = pd.read_json(data_path, lines=True)
sentences = [list(sentence) for sentence in input_sentences["word_tokens"] if sentence]
if len(sentences) < 3_000:
    raise ValueError("The corpus must contain at least 3,000 non-empty sentences.")

shuffled = pd.Series(range(len(sentences))).sample(frac=1, random_state=42).tolist()
test = [sentences[index] for index in shuffled[:1_000]]
dev = [sentences[index] for index in shuffled[1_000:2_000]]
train = [sentences[index] for index in shuffled[2_000:]]

print(f"Loaded {len(sentences):,} sentences from {data_path}")
print(f"Train: {len(train):,} | Dev: {len(dev):,} | Test: {len(test):,}")

Loaded 100,000 sentences from e:\STUDY\NLP\Lab 1\outputs\tokenized_sentences_with_source.txt
Train: 98,000 | Dev: 1,000 | Test: 1,000


In [15]:
from collections import Counter
from math import exp, log

class NGramLanguageModel:
    """
    N-Gram Language Model with flexible smoothing techniques.
    """
    START = "<s>"
    END = "</s>"

    def __init__(self, n):
        if not isinstance(n, int) or n < 1:
            raise ValueError("n must be a positive integer")
        self.n = n
        self.ngram_counts = Counter()
        self.context_counts = Counter()
        self.vocabulary = set()
        self.is_fitted = False
        # Pre-calculated counts for Good-Turing efficiency
        self.N_r = Counter() 
        self.total_tokens = 0

    def _padded(self, sentence):
        tokens = list(sentence)
        return [self.START] * (self.n - 1) + tokens + [self.END]

    def fit(self, corpus):
        """Count n-grams and their contexts from an iterable of token lists."""
        self.ngram_counts.clear()
        self.context_counts.clear()
        self.vocabulary.clear()
        self.N_r.clear()

        for sentence in corpus:
            padded = self._padded(sentence)
            self.vocabulary.update(padded)
            for index in range(len(padded) - self.n + 1):
                ngram = tuple(padded[index:index + self.n])
                context = ngram[:-1]
                self.ngram_counts[ngram] += 1
                self.context_counts[context] += 1

        # PERFORMANCE FIX: Pre-calculate Good-Turing N_r table and total tokens once
        # This prevents O(Vocabulary) computation inside every probability() call.
        self.total_tokens = sum(self.ngram_counts.values())
        for count in self.ngram_counts.values():
            self.N_r[count] += 1

        self.is_fitted = True
        return self

    def get_ml_prob(self, ngram):
        """Return the Maximum Likelihood Estimation (MLE) probability."""
        ngram = tuple(ngram)
        context = ngram[:-1]
        denominator = self.context_counts[context]
        return self.ngram_counts[ngram] / denominator if denominator else 0.0

    def probability(self, ngram, smoothing=None, **kwargs):
        """Main entry point for calculating probabilities."""
        if not self.is_fitted:
            raise RuntimeError("Fit the model before requesting probabilities")
        
        if smoothing is None:
            return self.get_ml_prob(ngram)

        smoothing_map = {
            'interpolated': self.smooth_interpolated,
            'good_turing': self.smooth_good_turing,
            'katz': self.smooth_katz,
            'stupid': self.smooth_stupid,
            'kneser_ney': self.smooth_kneser_ney,
        }

        if smoothing not in smoothing_map:
            raise ValueError(f"Unknown smoothing technique: {smoothing}")

        return smoothing_map[smoothing](ngram, **kwargs)

    # --- Smoothing Implementations ---

    def smooth_interpolated(self, ngram, **kwargs):
        lower_order_models = kwargs.get('lower_order_models', [])
        lambdas = kwargs.get('lambdas', [])
        if not lower_order_models or not lambdas:
            return self.get_ml_prob(ngram)
        prob = lambdas[0] * self.get_ml_prob(ngram)
        for i, lower_model in enumerate(lower_order_models):
            shorter_ngram = ngram[i+1:]
            prob += lambdas[i+1] * lower_model.get_ml_prob(shorter_ngram)
        return prob

    def smooth_good_turing(self, ngram, **kwargs):
        """Optimized Good-Turing Smoothing using pre-calculated N_r."""
        context = tuple(ngram)[:-1]
        deno_context = self.context_counts[context]
        if deno_context == 0: return 0.0

        c = self.ngram_counts[ngram]
        if c == 0:
            # P = N_1 / N
            return self.N_r[1] / self.total_tokens if self.total_tokens > 0 else 0.0
        else:
            # Smoothed count = ((c+1) * N_{c+1}) / N_c
            n_c_plus_1 = self.N_r[c + 1]
            n_c = self.N_r[c]
            if n_c == 0 or n_c_plus_1 == 0:
                return self.get_ml_prob(ngram)
            return ((c + 1) * n_c_plus_1 / n_c) / deno_context

    def smooth_katz(self, ngram, **kwargs):
        lower_order_models = kwargs.get('lower_order_models', [])
        if self.ngram_counts[ngram] > 0:
            return self.smooth_good_turing(ngram, **kwargs)
        if not lower_order_models:
            return self.get_ml_prob(ngram)
        shorter_ngram = ngram[1:]
        return 0.5 * lower_order_models[0].probability(shorter_ngram, smoothing='katz', lower_order_models=lower_order_models[1:])

    def smooth_stupid(self, ngram, **kwargs):
        lower_order_models = kwargs.get('lower_order_models', [])
        alpha = 0.4
        if self.ngram_counts[ngram] > 0:
            return self.get_ml_prob(ngram)
        if not lower_order_models:
            return self.get_ml_prob(ngram)
        shorter_ngram = ngram[1:]
        return alpha * lower_order_models[0].probability(shorter_ngram, smoothing='stupid', lower_order_models=lower_order_models[1:])

    def smooth_kneser_ney(self, ngram, **kwargs):
        lower_order_models = kwargs.get('lower_order_models', [])
        delta = 0.75
        count = self.ngram_counts[ngram]
        context_count = self.context_counts[tuple(ngram)[:-1]]
        if count > 0 and context_count > 0:
            return (count - delta) / context_count
        if not lower_order_models:
            return self.get_ml_prob(ngram)
        shorter_ngram = ngram[1:]
        return lower_order_models[0].probability(shorter_ngram, smoothing='kneser_ney', lower_order_models=lower_order_models[1:])

# Fit the 4 models
unigram = NGramLanguageModel(n=1).fit(train)
bigram = NGramLanguageModel(n=2).fit(train)
trigram = NGramLanguageModel(n=3).fit(train)
quadgram = NGramLanguageModel(n=4).fit(train)

print("Models fitted successfully.")


Models fitted successfully.


In [16]:
import pandas as pd
import numpy as np

def calculate_perplexity(model, dataset, smoothing=None, **kwargs):
    """Calculate perplexity of a model on a given dataset."""
    total_log_prob = 0
    total_words = 0
    
    for sentence in dataset:
        padded = model._padded(sentence)
        # Start from model.n - 1 to ensure the first word of the sentence is predicted
        for i in range(model.n - 1, len(padded)):
            # Extract n-gram ending at index i
            ngram = tuple(padded[i - model.n + 1 : i + 1])
            
            # Basic safety check: skip if ngram is too short (shouldn't happen with padding)
            if len(ngram) < model.n:
                continue
                
            prob = model.probability(ngram, smoothing=smoothing, **kwargs)
            
            # Use a small epsilon to avoid log(0) and avoid infinity
            prob = max(prob, 1e-12)
            total_log_prob += log(prob)
            total_words += 1
            
    if total_words == 0: return float('inf')
    return exp(-total_log_prob / total_words)

# Define models and their lower-order dependencies for smoothing
model_configs = [
    {'name': 'Unigram', 'model': unigram, 'lower': [], 'lambdas': [1.0]},
    {'name': 'Bigram', 'model': bigram, 'lower': [unigram], 'lambdas': [0.7, 0.3]},
    {'name': 'Trigram', 'model': trigram, 'lower': [bigram, unigram], 'lambdas': [0.7, 0.2, 0.1]},
    {'name': 'Quadgram', 'model': quadgram, 'lower': [trigram, bigram, unigram], 'lambdas': [0.7, 0.2, 0.05, 0.05]},
]

smoothing_techniques = ['interpolated', 'good_turing', 'katz', 'stupid', 'kneser_ney']
results = []

for config in model_configs:
    for tech in smoothing_techniques:
        # Setup kwargs for this specific model/technique
        kwargs = {
            'lower_order_models': config['lower'],
            'lambdas': config['lambdas']
        }
        
        pp_dev = calculate_perplexity(config['model'], dev, smoothing=tech, **kwargs)
        pp_test = calculate_perplexity(config['model'], test, smoothing=tech, **kwargs)
        
        results.append({
            'Model': config['name'],
            'Smoothing': tech,
            'Dev Perplexity': pp_dev,
            'Test Perplexity': pp_test
        })

# Display results as a DataFrame
df_results = pd.DataFrame(results)
# Use to_string() for full display in terminal/notebook
print("Evaluation Results (Perplexity):")
print(df_results.to_string(index=False))

# Store the final results in a CSV file in Lab 6 directory
output_path = "E:/STUDY/NLP/Lab 6/smoothing_results.csv"
df_results.to_csv(output_path, index=False)
print(f"\nResults successfully saved to: {output_path}")


Evaluation Results (Perplexity):
   Model    Smoothing  Dev Perplexity  Test Perplexity
 Unigram interpolated    1.057355e+04     1.086603e+04
 Unigram  good_turing    2.957252e+03     2.997958e+03
 Unigram         katz    1.141811e+04     1.177336e+04
 Unigram       stupid    1.057355e+04     1.086603e+04
 Unigram   kneser_ney    1.122550e+04     1.155943e+04
  Bigram interpolated    3.821124e+03     3.656910e+03
  Bigram  good_turing    6.282178e+01     6.504541e+01
  Bigram         katz    3.562624e+03     3.439396e+03
  Bigram       stupid    2.960740e+03     2.830517e+03
  Bigram   kneser_ney    2.536381e+03     2.458219e+03
 Trigram interpolated    6.943587e+03     6.406536e+03
 Trigram  good_turing    8.286668e+05     7.153949e+05
 Trigram         katz    5.561687e+03     5.228436e+03
 Trigram       stupid    4.675679e+03     4.320391e+03
 Trigram   kneser_ney    2.266200e+03     2.138036e+03
Quadgram interpolated    1.502511e+04     1.371833e+04
Quadgram  good_turing    1.20666

In [17]:
evaluation = []
for config in model_configs:
    model = config["model"]
    evaluation.append({
        "model": config["name"],
        "order": model.n,
        "vocabulary_size": len(model.vocabulary),
        "dev_perplexity": calculate_perplexity(model, dev, smoothing="interpolated", lower_order_models=config["lower"], lambdas=config["lambdas"]),
        "test_perplexity": calculate_perplexity(model, test, smoothing="interpolated", lower_order_models=config["lower"], lambdas=config["lambdas"]),
    })

results = pd.DataFrame(evaluation)
display(results)
assert len(results) == 4
assert results[["dev_perplexity", "test_perplexity"]].notna().all().all()


,model,order,vocabulary_size,dev_perplexity,test_perplexity
0,Unigram,1,139533,10573.550960,10866.030797
1,Bigram,2,139534,3821.124337,3656.909801
2,Trigram,3,139534,6943.587291,6406.535646
3,Quadgram,4,139534,15025.107487,13718.334183
